# IndicatorsEnv v3.0 — Colab Test Notebook

Test the portfolio MDP environment and run inference with a small HF model.

**What this notebook does:**
1. Upload & extract the hackathon code
2. Install dependencies
3. Start the IndicatorsEnv server (FastAPI on port 7860)
4. Smoke test — verify episode structure, grader, and baseline
5. Full inference — run `inference.py` with a 1B model against the live environment

**IndicatorsEnv v3.0 — Portfolio MDP:**
- Agent manages a long/short/flat position in a single NSE stock
- Observation includes indicator snapshot + portfolio state (position, unrealized_pnl, capital)
- Reward = actual next-day return × position − 0.1% transaction cost per trade
- Episode lengths: short=5 steps, medium=10 steps, long=20 steps
- Task 3: early termination if drawdown > 5%, macro context added to observation

## Step 1: Upload & Extract Code

Create the zip on your Mac first, then upload it here:

```bash
# Run this in your terminal (Mac)
cd "/Users/aaryanmanawat/Aaryan/StockAnalyzer Pro/version3.0/StockAnalyzer-Pro"
zip -r hackathon.zip hackathon/env hackathon/inference.py hackathon/requirements.txt
```

Then run the cell below and select `hackathon.zip` when the file picker appears.

In [ ]:
from google.colab import files
import zipfile, os

uploaded = files.upload()  # Select hackathon.zip from your Mac

with zipfile.ZipFile('hackathon.zip', 'r') as z:
    z.extractall('/content/')

%cd /content/hackathon
print(f'Ready in: {os.getcwd()}')
print('Files:', os.listdir('.'))

## Step 2: Install Dependencies

In [ ]:
!pip install -q fastapi uvicorn yfinance pandas numpy requests openai
print('Done')

## Step 3: Start the IndicatorsEnv Server

Starts FastAPI on `http://localhost:7860`.
The first `reset()` call takes ~10–30s as yfinance fetches real NSE OHLCV data.

> **Note:** Uses `workers=1` — required because session state is in-memory.

In [ ]:
import subprocess, time, sys, requests

server = subprocess.Popen(
    [sys.executable, 'indicators_env.py'],
    cwd='/content/hackathon/env',
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
time.sleep(12)

try:
    resp = requests.get('http://localhost:7860/health', timeout=5)
    h = resp.json()
    print('Server:', h)
    print('Episode lengths:', h.get('episode_steps', {}))
except Exception as e:
    print('Not ready:', e)
    out = server.stdout.read1(2048).decode('utf-8', errors='replace')
    print('Startup log:', out)

## Step 4: Smoke Test (No LLM Required)

Verifies four things:
- **Multi-step structure**: `done=False` on steps 1–4, `done=True` on step 5 (short task)
- **Portfolio state**: observation contains `position`, `unrealized_pnl`, `capital_remaining`
- **Grader**: returns `score > 0` with the `"predicted"` key
- **Baseline**: short task gets 50 steps (5 steps × 10 episodes), score ≈ 0.33

In [ ]:
import requests

BASE = 'http://localhost:7860'

# ── Reset (short task = 5 steps) ──────────────────────────────────────────────
r = requests.post(f'{BASE}/reset', params={'term': 'short'}, timeout=60)
assert r.status_code == 200, f'Reset failed: {r.text}'
data       = r.json()
session_id = data['info']['session_id']
obs        = data['observation']
print(f'Reset OK — symbol={obs["symbol"]}  date={obs["date"]}')
print(f'Portfolio state: position={obs["position"]}, '
      f'unrealized_pnl={obs["unrealized_pnl"]}, '
      f'capital_remaining={obs["capital_remaining"]}')
print(f'Session: {session_id}\n')

# ── 5 Steps ───────────────────────────────────────────────────────────────────
print('Steps:')
for i in range(1, 6):
    s = requests.post(
        f'{BASE}/step',
        params={'session_id': session_id},
        json={'direction': 'Bullish', 'conviction': 0.7},
        timeout=30,
    ).json()
    step   = s['info']['step']
    done   = s['done']
    reward = s['reward']
    actual = s['info'].get('actual_return_pct', 'N/A')
    capital = s['info'].get('capital', 'N/A')
    ok     = (i < 5 and not done) or (i == 5 and done)
    print(f'  step={step}  done={done}  reward={reward:.4f}  '
          f'actual_return={actual}%  capital={capital}  {"✅" if ok else "❌ WRONG"}')

# ── Grader ────────────────────────────────────────────────────────────────────
print()
g = requests.post(f'{BASE}/grader', json={
    'task_id': 'short_term_direction',
    'episode_results': [
        {'predicted': 'Bullish', 'ground_truth': 'Bullish', 'conviction': 0.7},
        {'predicted': 'Bearish', 'ground_truth': 'Bearish', 'conviction': 0.8},
        {'predicted': 'Bullish', 'ground_truth': 'Neutral',  'conviction': 0.5},
    ],
}).json()
score = g['score']
print(f'Grader score: {score}  {"✅" if score > 0 else "❌ WRONG — grader bug"}')

# ── Baseline (takes ~60s with real yfinance) ──────────────────────────────────
print('\nRunning baseline — this takes ~60s...')
b = requests.get(f'{BASE}/baseline', timeout=300).json()
short_eps = b['tasks']['short_term_direction']['num_episodes']
mean      = b['overall_mean']
ok        = short_eps == 50   # short: 5 steps × 10 episodes
print(f'short_term num_episodes={short_eps}  {"✅" if ok else f"❌ WRONG (expected 50)"}')
print(f'medium_term num_episodes={b["tasks"]["medium_term_direction"]["num_episodes"]}  (expect 100)')
print(f'long_term   num_episodes={b["tasks"]["long_term_conviction"]["num_episodes"]}  (expect ≤200, may be less if drawdown triggered)')
print(f'overall_mean={mean}  (expect ~0.33 for random agent)')

## Step 5: Full Inference with a 1B Model

Runs `inference.py` end-to-end against the live environment.
Uses `meta-llama/Llama-3.2-1B-Instruct` via the HF Inference API (free tier).

**Paste your HF token below.** `--n_episodes 1` = 15 total LLM calls (5+10+20 steps across 3 tasks).

> Get your token at: https://huggingface.co/settings/tokens (read access is enough)

In [ ]:
import os

# ── Paste your HF token below ─────────────────────────────────────────────────
HF_TOKEN = 'hf_YOUR_TOKEN_HERE'   # ← replace this
# ─────────────────────────────────────────────────────────────────────────────

assert HF_TOKEN != 'hf_YOUR_TOKEN_HERE', 'Replace HF_TOKEN with your real token first!'

os.environ['API_BASE_URL'] = 'https://router.huggingface.co/v1/'
os.environ['MODEL_NAME']   = 'meta-llama/Llama-3.2-1B-Instruct'
os.environ['HF_TOKEN']     = HF_TOKEN

!python /content/hackathon/inference.py \
    --env_url http://localhost:7860 \
    --n_episodes 1

### Expected Output

```
[START] task=short_term_direction env=IndicatorsEnv model=meta-llama/Llama-3.2-1B-Instruct
[STEP]  step=1 action=Bullish reward=0.3500 done=False error=None
[STEP]  step=2 action=Neutral  reward=0.0000 done=False error=None
[STEP]  step=3 action=Bullish  reward=0.2100 done=False error=None
[STEP]  step=4 action=Bearish  reward=-0.1200 done=False error=None
[STEP]  step=5 action=Bullish  reward=0.4500 done=True  error=None
[END]   success=True steps=5 score=0.XXXX rewards=[...]

[START] task=medium_term_direction ...
... (10 [STEP] lines) ...
[END]   success=True steps=10 ...

[START] task=long_term_conviction ...
... (up to 20 [STEP] lines, may end early if drawdown > 5%) ...
[END]   success=True steps=≤20 ...
```

**What to check:**
- `done=False` on steps 1–4, `done=True` on step 5 for short task
- `steps=5` for short, `steps=10` for medium, `steps≤20` for long
- `score > 0` in every `[END]` line
- Rewards are non-integer floats (actual market returns × position × 50 scale)
- 3 tasks × 5+10+20 = 35 total `[STEP]` lines per `n_episodes=1` run